## **Assignment 2: Modeling and Solving Linear Programming Problems**

**Author** : Rakshitha Vignesh Sargurunathan (V01109007)

**Date**   : 09/16/2024

## Blaire and Rosen Inc

In [ ]:
# Install dependencies
!pip install -q amplpy ampltools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 18.7 MB/s eta 0:00:00


In [ ]:
# Google Colab & AMPL integration
MODULES, LICENSE_UUID = ["coin", 'gurobi', "cplex", "highs", "gokestrel"], "42fc7eb6-69aa-445d-b655-3ad24d836541"
from amplpy import tools
from ampltools import cloud_platform_name, ampl_notebook, register_magics

# instantiate AMPL object and register magics
if cloud_platform_name() is None:
    ampl = AMPL() # Use local installation of AMPL
else:
    ampl = tools.ampl_notebook(modules=MODULES, license_uuid=LICENSE_UUID, g=globals())

register_magics(ampl_object=ampl)

Licensed to Bundle #6741.7193 expiring 20241231: INFO 645 Prescriptive Analytics, Prof. Paul Brooks, Virginia Commonwealth University.


In [ ]:
ampl.eval ('''
reset;

# Sets
set F;  # Set of investment funds

# Parameters
param return {i in F};   # Projected annual return for each fund
param risk {i in F};     # Risk rating per $1,000 invested in each fund
param total_investment;  # Total amount available to invest
param max_investment_Internet;  # Maximum allowable investment in Internet fund
param max_risk;  # Maximum allowable risk rating for the portfolio

# Variables
var x {i in F} >= 0;   # Investment in each fund (non-negative)

# Objective: Maximize total return
maximize total_return: sum{i in F} return[i] * x[i];

# Constraints
subject to
total_investment_constraint: sum{i in F} x[i] <= total_investment;
max_investment_constraint: x['Internet'] <= max_investment_Internet;
risk_constraint: sum{i in F} risk[i] * (x[i] / 1000) <= max_risk;

''')


In [ ]:
# Define the data for the portfolio problem

ampl.set['F'] = ['Internet', 'Blue Chip']  # Set of investment funds

# Define the parameters
ampl.param['return'] = {'Internet': 0.12, 'Blue Chip': 0.09}  # Projected annual return for each fund
ampl.param['risk'] = {'Internet': 6, 'Blue Chip': 4}  # Risk rating per $1,000 invested in each fund

# Total available investment and limits
ampl.param['total_investment'] = 50000  # Total amount available to invest ($50,000)
ampl.param['max_investment_Internet'] = 35000  # Maximum investment in the Internet fund ($35,000)
ampl.param['max_risk'] = 240  # Maximum allowable risk rating (240)

In [ ]:
ampl.eval('''expand;''')

maximize total_return:
	0.12*x['Internet'] + 0.09*x['Blue Chip'];

subject to total_investment_constraint:
	x['Internet'] + x['Blue Chip'] <= 50000;

subject to max_investment_constraint:
	x['Internet'] <= 35000;

subject to risk_constraint:
	0.006*x['Internet'] + 0.004*x['Blue Chip'] <= 240;



In [ ]:
ampl.setOption('solver', 'gurobi')
ampl.solve()

Gurobi 11.0.3: Gurobi 11.0.3: optimal solution; objective 5100
0 simplex iterations


In [ ]:
# Display the investment results
investment_internet = ampl.var['x']['Internet'].value()
investment_bluechip = ampl.var['x']['Blue Chip'].value()
total_return = ampl.obj['total_return'].value()

print(f'Investment in Internet fund: ${investment_internet:,.2f}')
print(f'Investment in Blue Chip fund: ${investment_bluechip:,.2f}')
print(f'Total Return: ${total_return:,.2f}')

Investment in Internet fund: $20,000.00
Investment in Blue Chip fund: $30,000.00
Total Return: $5,100.00


##RESULT:

The optimal solution is to invest 20,000 dollars in the Internet Fund and 30,000 dollars in the Blue Chip Fund for a total return of $5,100 .